# Cataract Detection using VGG19 Transfer Learning

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os

import tensorflow as tf
from tensorflow.keras.applications import VGG19
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 2. Configuration and Reproducibility

> **Note:** Define `SEED`, `DATA_DIR`, `CLASSES`, `IMG_SIZE`, `EPOCHS`, and `BATCH_SIZE` here before running the rest of the notebook.

In [ ]:
SEED = 42
DATA_DIR = 'data'
CLASSES = ['normal', 'cataract']
IMG_SIZE = 224
EPOCHS = 30
BATCH_SIZE = 32

np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
print('All libraries loaded successfully!')

## 3. Load Dataset

In [ ]:
def load_dataset(data_dir, classes, img_size):
    images, labels = [], []
    for label_idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        file_list = sorted(os.listdir(class_dir))
        count = 0
        for fname in file_list:
            fpath = os.path.join(class_dir, fname)
            img = cv2.imread(fpath)        # Load as BGR
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))  # Resize to 224x224
            images.append(img)
            labels.append(label_idx)
            count += 1
        print(f'Loaded {count} images for class: {class_name}')
    return np.array(images), np.array(labels)

X, y = load_dataset(DATA_DIR, CLASSES, IMG_SIZE)
print()
print('Total images:', X.shape[0])
print('Image shape (H x W x Channels):', X.shape[1:])

## 4. Visualize Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Sample Retinal Fundus Images', fontsize=14, fontweight='bold')

for i, c in enumerate(CLASSES):
    idxs = np.where(y == i)[0][:4]
    for j, idx in enumerate(idxs):
        ax = axes[i, j]
        ax.imshow(cv2.cvtColor(X[idx], cv2.COLOR_BGR2RGB))
        ax.set_title(c.capitalize(), fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

## 5. Preprocess Data (Normalize, One-Hot Encode, Train/Test Split)

In [ ]:
# Normalise pixel values from [0,255] to [0,1]
X_norm = X.astype('float32') / 255.0

# One-hot encode labels
y_cat = to_categorical(y, num_classes=len(CLASSES))

# 80% train, 20% test — stratified so both classes are balanced in each split
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_cat,
    test_size=0.20,
    stratify=y,
    random_state=SEED
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## 6. Build VGG19 Transfer Learning Model

In [ ]:
# Load VGG19 pretrained on ImageNet, without the top classification layers
base_model = VGG19(weights='imagenet',
                   include_top=False,
                   input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Freeze all convolutional layers — we only train our new classification head
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(CLASSES), activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1)

print('\nTraining complete!')

## 8. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['accuracy'],     marker='o', label='Train',      color='green')
axes[0].plot(history.history['val_accuracy'], marker='o', label='Validation', color='red')
axes[0].set_title('Model Accuracy', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     marker='o', label='Train',      color='green')
axes[1].plot(history.history['val_loss'], marker='o', label='Validation', color='red')
axes[1].set_title('Model Loss', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_loss_curves.png')

## 9. Evaluate on Test Set

In [ ]:
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test,       axis=1)

test_acc = accuracy_score(y_true, y_pred)

print('=' * 50)
print(f'TEST ACCURACY: {test_acc:.4f}  ({test_acc*100:.2f}%)')
print('=' * 50)
print()
print('CLASSIFICATION REPORT:')
print(classification_report(y_true, y_pred, target_names=CLASSES))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=CLASSES,
            yticklabels=CLASSES,
            linewidths=0.5)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label',      fontsize=11)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')